In [51]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, AIMessage

load_dotenv(override=True)
checkpointer = InMemorySaver()

In [34]:
openai_key = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=openai_key,
    temperature=0.2,
    max_tokens=None,
    max_retries=2
)
llm2 = ChatOpenAI(
    model="gpt-4.1-nano",
    api_key=openai_key,
    temperature=0.2,
    max_tokens=None,
    max_retries=2
)

In [1]:
topics = [{
    "name": "Small Business Defined",
    "notes": f"""Small businesses are often the starting point for entrepreneurs as they develop their ideas and build a customer base. 
    The Small Business Administration (SBA) defines a small business as a for-profit entity with fewer than 500 employees. This definition makes
    these businesses eligible for various government programs and preferences. Small businesses play a crucial role in our economy and communities."""
    },
    {
    "name": "Small Business Impact",
    "notes": f"""There are over 33.2 million small businesses in the United States, making up 99.9% of all firms. From 1995 to 2021, small businesses created 
    17.3 million net new jobs, significantly more than large businesses. Despite challenges like the COVID-19 recession, small businesses rebounded quickly,
    demonstrating their resilience and importance to economic recovery. They contribute to local economies by reinvesting paychecks and taxes, supporting 
    new businesses, and improving
    public services. On average, small businesses offer competitive wages, averaging $30.42 per hour, translating to an annual income of $63,000."""
    },
    {
    "name": "Small Business Demographics",
    "notes": f"""43.4% of small businesses are owned by females, reflecting progress toward gender equality in entrepreneurship.
    20.4% are owned by racial minorities, including 14.5% by Hispanics.
    6.1% are owned by veterans, contributing diverse perspectives to the U.S. economy.
    Hispanic-owned businesses, for example, pay over $100 billion annually in payroll to their 1 million workers."""
    }
    ]

In [70]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, MessagesState

class TopicState(MessagesState):
    title: str
    notes: str
    question: str
    reply: str
    hint_taken: bool
    

In [71]:
class DialogueState(MessagesState):
    index: int
    max_ind: int
    question: TopicState
    topics: list
    

In [56]:
dialogues: DialogueState = {
    "index": -1,    
    "topics": [],
    "topic": {
    }
}

In [57]:
def initialize_graph(state: DialogueState) -> DialogueState:
    state["topics"] = topics
    state["max_ind"] = len(topics)
    return state

In [18]:
def set_index(state: DialogueState) -> DialogueState:
    state["index"] += 1

In [21]:
from typing import Literal

def check_termination(state: DialogueState) -> Literal["continue", "terminate"]:
    return "terminate" if state["index"] >= state["max_ind"] else "continue"

In [58]:
def set_question(state: DialogueState) -> DialogueState:
    question_topic = state["topics"][state["index"]]
    topic, notes, hint_taken = question_topic["title"], question_topic["notes"], False

    question_prompt = f"""
    You are an examiner whose job is to prepare a conceptual question on a topic using the notes provided to you.
    Topic: {topic}
    Notes: {notes}
    Keep the question simple, just to test a basic understanding.
    """
    question = llm.invoke(question_prompt).content
    
    state["topic"] = {
        "topic": topic,
        "notes": notes,
        "question": question,
        "hint_taken": hint_taken,
        "reply": ""
    }
    return state
    

In [59]:
def get_student_answer(state: DialogueState):
    user_answer = input(state["topic"]["question"])
    state["topic"]["reply"] = user_reply
    return state
    

In [39]:
from pydantic import BaseModel

class EvalSchema(BaseModel):
    evaluation: Literal["satisfactory", "unsatisfactory"]
    comment: str

In [64]:
def evaluate(state: DialogueState) -> Literal["hint", "satisfactory", "unsatisfactory"]:
    
    give_hint = False if state["topic"]["hint_taken"] else True
    """
    Evaluate the user reply, using LLM.
    Returns correct, retry or limits reached outputs
    """
    if give_hint:
        ai_prompt = f"""
            You have to analyze the user's reply to a question to check the understanding of a concept and tell whether
            it is acceptable using notes provided to you. 
            Return: satisfactory or unsatisfactory, along with some comment on the users answer. 
            If not satisfactory, give hint in the comment
            without giving complete answer.
            Keep user as the first person and address the answer to user only.
            Question: {state["topic"]["question"]}
            Notes: {state["topic"]["notes"]}
            User's reply: {state["topic"]["reply"]}
        """
    else:
        ai_prompt = f"""
            You have to analyze the user's reply to a question to check the understanding of a concept and tell whether
            it is acceptable using notes provided to you. 
            Return: satisfactory or unsatisfactory, along with some comment on the users answer. 
            If not satisfactory, give answer using notes.
            Keep user as the first person and address the answer to user only.
            
            Question: {state["topic"]["question"]}
            Notes: {state["topic"]["notes"]}
            User's reply: {state["topic"]["reply"]}
        """
    
    llm_s = llm.with_structured_output(EvalSchema)
    evaluation = llm_s.invoke(ai_prompt)
    state["topic"]["messages"].append(AIMessage(content=evaluation.comment))
    
    if evaluation.evaluation == "satisfactory":        
        return "satisfactory"
    else:
        if give_hint:
            state["topic"]["hint_taken"] = True
            return "hint"
        else:
            return "unsatisfactory"
    
    

In [60]:
def satisfactory(state):
    user_input = input(state["topic"]["messages"][-1].content)
    return state

In [ ]:
def hint(state):
    user_input = input(state["topic"]["messages"][-1].content)
    state["topic"]["reply"] = user_input
    return state

In [73]:
def unsatisfactory(state):
    user_input = input(state["topic"]["messages"][-1].content)
    return state

In [75]:
s = {
    "index": 0,    
    "topics": topics,
    "topic": {
        "topic": "Small business defined",
        "notes": """Small businesses are often the starting point for entrepreneurs as they develop their ideas and build a customer base. 
    The Small Business Administration (SBA) defines a small business as a for-profit entity with fewer than 500 employees. This definition makes
    these businesses eligible for various government programs and preferences. Small businesses play a crucial role in our economy and communities.""",
        "question": "How do you define small business",
        "hint_taken": False,
        "messages": [],
        "reply": """Small businesses are often the starting point for entrepreneurs as they develop their ideas and build a customer base. 
    The Small Business Administration (SBA) defines a small business as a for-profit entity with fewer than 500 employees. This definition makes
    these businesses eligible for various government programs and preferences. Small businesses play a crucial role in our economy and communities"""
    }
}
evaluate(s)

'satisfactory'